In [1]:
# mount from drive
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Readthrough_project/mRNA-LM-Readthrough//

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1aSW3hGEKww8b1esY_ea6xWFj73ysLZe8/Readthrough_project/mRNA-LM-original


In [ ]:
!python /content/drive/MyDrive/Readthrough_project/mRNA-LM-Readthrough/finetune_all.py --task readthrough --output /content/drive/MyDrive/Readthrough_project/test_output_new

2025-09-03 11:42:32.803585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756899752.824210    2346 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756899752.830380    2346 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756899752.845609    2346 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756899752.845635    2346 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756899752.845638    2346 computation_placer.cc:177] computation placer alr

In [ ]:
# ===== ALL ESSENTIAL CODE IN ONE CELL =====

# 1. IMPORTS

"""
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import TrainingArguments, Trainer, BertForMaskedLM, PreTrainedTokenizerFast
from transformers.activations import gelu
from scipy.stats import pearsonr, spearmanr
from peft import LoraConfig, TaskType, get_peft_model
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import BertProcessing

os.environ["TOKENIZERS_PARALLELISM"] = "false" # Set to false for this test

# 2. DATA LOADING LOGIC (from dataload.py)
def mytok(seq, kmer_len, s):
    seq = seq.upper().replace("T", "U")
    kmer_list = []
    for j in range(0, (len(seq) - kmer_len) + 1, s):
        kmer_list.append(seq[j:j + kmer_len])
    return kmer_list

def build_dp_dataset():
    def load_dataset_internal(data_path, split):
        df = pd.read_csv(data_path)
        df = df[df["split"] == split]
        df = df.dropna(subset=["y"])

        utr5 = df["UTR5"].values.tolist()
        utr3 = df["UTR3"].values.tolist()
        cds = df["CDS"].values.tolist()
        ys = df["y"].values.tolist()

        utr5 = [" ".join(mytok(seq, 1, 1)) for seq in utr5]
        cds  = [" ".join(mytok(seq, 3, 3)) for seq in cds]
        utr3 = [" ".join(mytok(seq, 1, 1)) for seq in utr3]
        seqs = list(zip(utr5, cds, utr3))

        return seqs, ys

    train_seqs, train_ys = load_dataset_internal("/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/translation_rate.csv", "train")
    valid_seqs, valid_ys = load_dataset_internal("/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/translation_rate.csv", "valid")

    ds_train = Dataset.from_list([{"5utr": seq[0], "cds": seq[1], "3utr": seq[2], "label": y} for seq, y in zip(train_seqs, train_ys)])
    ds_valid = Dataset.from_list([{"5utr": seq[0], "cds": seq[1], "3utr": seq[2], "label": y} for seq, y in zip(valid_seqs, valid_ys)])

    return ds_train, ds_valid

# 3. MODEL DEFINITION (from FullModel.py)
class FullModel(torch.nn.Module):
    def __init__(self, num_labels, lorar, lalpha, ldropout, head_dim, head_droupout):
        super(FullModel, self).__init__()
        self.tokenizer_cds, self.tokenizer_5utr, self.tokenizer_3utr = self.build_tokenizer()

        self.utr5 = BertForMaskedLM.from_pretrained("/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/mrna_5utr_model")
        self.utr3 = BertForMaskedLM.from_pretrained("/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/mrna_3utr_model")
        self.cds = BertForMaskedLM.from_pretrained("/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/CodonBERT")

        if lorar > 0:
            peft_config = LoraConfig(r=lorar, lora_alpha=lalpha, lora_dropout=ldropout, task_type="TOKEN_CLS")
            self.utr5 = get_peft_model(self.utr5, peft_config)
            self.utr3 = get_peft_model(self.utr3, peft_config)
            self.cds = get_peft_model(self.cds, peft_config)

        self.final_dense = nn.Linear(768 * 3, head_dim)
        self.transform_act_fn = gelu
        self.LayerNorm = torch.nn.LayerNorm(head_dim, eps=1e-12)
        self.dropout = nn.Dropout(head_droupout)
        self.decoder = nn.Linear(head_dim, num_labels)
        self.loss_fn = nn.MSELoss()

    def get_mean_token_embeddings(self, token_embeddings, token_mask):
        input_mask_expanded = token_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return torch.sum(token_embeddings * input_mask_expanded, 1) / sum_mask

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, labels, **kwargs):
        utr5_embeds = self.utr5(input_ids=input_ids1, attention_mask=attention_mask1).hidden_states[-1]
        cds_embeds  = self.cds(input_ids=input_ids2, attention_mask=attention_mask2).hidden_states[-1]
        utr3_embeds = self.utr3(input_ids=input_ids3, attention_mask=attention_mask3).hidden_states[-1]

        utr5_sum = self.get_mean_token_embeddings(utr5_embeds[:, 1:-1, :], attention_mask1[:, 1:-1])
        cds_sum  = self.get_mean_token_embeddings(cds_embeds[:, 1:-1, :], attention_mask2[:, 1:-1])
        utr3_sum = self.get_mean_token_embeddings(utr3_embeds[:, 1:-1, :], attention_mask3[:, 1:-1])

        joint_embed = torch.cat([utr5_sum, cds_sum, utr3_sum], dim=1)
        hidden_states = self.dropout(self.LayerNorm(self.transform_act_fn(self.final_dense(joint_embed))))
        logits = self.decoder(hidden_states).squeeze()
        loss = self.loss_fn(logits, labels.float())

        return loss, logits

    def build_tokenizer(self):
        # Your build_tokenizer logic here... (shortened for brevity, use your full version)
        # CDS Tokenizer
        lst_ele = list('AUGCN')
        lst_voc_cds = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'] + [f'{a1}{a2}{a3}' for a1 in lst_ele for a2 in lst_ele for a3 in lst_ele]
        dic_voc_cds = {t: i for i, t in enumerate(lst_voc_cds)}
        tok_cds_obj = Tokenizer(WordLevel(vocab=dic_voc_cds, unk_token="[UNK]"))
        tok_cds_obj.pre_tokenizer = Whitespace()
        tok_cds_obj.post_processor = BertProcessing(("[SEP]", dic_voc_cds['[SEP]']), ("[CLS]", dic_voc_cds['[CLS]']))
        tokenizer_cds = PreTrainedTokenizerFast(tokenizer_object=tok_cds_obj, model_max_length=1024, pad_token='[PAD]', cls_token='[CLS]', sep_token='[SEP]', unk_token='[UNK]', mask_token='[MASK]')

        # UTR Tokenizer
        lst_voc_utr = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'] + list('AUGCN')
        dic_voc_utr = {t: i for i, t in enumerate(lst_voc_utr)}
        tok_utr_obj = Tokenizer(WordLevel(vocab=dic_voc_utr, unk_token="[UNK]"))
        tok_utr_obj.pre_tokenizer = Whitespace()
        tok_utr_obj.post_processor = BertProcessing(("[SEP]", dic_voc_utr['[SEP]']), ("[CLS]", dic_voc_utr['[CLS]']))
        tokenizer_utr = PreTrainedTokenizerFast(tokenizer_object=tok_utr_obj, model_max_length=512, pad_token='[PAD]', cls_token='[CLS]', sep_token='[SEP]', unk_token='[UNK]', mask_token='[MASK]')

        return tokenizer_cds, tokenizer_utr, tokenizer_utr # UTR5 and UTR3 use the same tokenizer

    def encode_string(self, data):
        tok_5utr = self.tokenizer_5utr(data['5utr'], truncation=True, padding="max_length", max_length=512)
        tok_cds = self.tokenizer_cds(data['cds'], truncation=True, padding="max_length", max_length=1024)
        tok_3utr = self.tokenizer_3utr(data['3utr'], truncation=True, padding="max_length", max_length=1024)
        return {
            'input_ids1': tok_5utr['input_ids'], 'attention_mask1': tok_5utr['attention_mask'],
            'input_ids2': tok_cds['input_ids'], 'attention_mask2': tok_cds['attention_mask'],
            'input_ids3': tok_3utr['input_ids'], 'attention_mask3': tok_3utr['attention_mask'],
            'labels': data['label']
        }

# 4. MAIN SCRIPT LOGIC (from finetune_all.py)

# Hardcoded arguments (bypassing argparse)
output_dir = "./test_output"
lorar = 32
lalpha = 32
ldropout = 0.5
lr = 1e-5
head_dim = 768
head_dropout = 0.5
batch_size = 8 # Use a small batch size for testing

print("--- Initializing Model ---")
model = FullModel(num_labels=1, lorar=lorar, lalpha=lalpha, ldropout=ldropout, head_dim=head_dim, head_droupout=head_dropout)

print("--- Building Dataset ---")
ds_train, ds_valid = build_dp_dataset()

# CRITICAL DEBUG STEP: Check the dataset columns BEFORE mapping
print("\n--- COLUMNS BEFORE .map() ---")
print(ds_train)
print("\n")


print("--- Mapping Dataset ---")
train_loader = ds_train.map(model.encode_string, batched=True)
val_loader = ds_valid.map(model.encode_string, batched=True)

# CRITICAL DEBUG STEP: Check the dataset columns AFTER mapping
print("\n--- COLUMNS AFTER .map() ---")
print(train_loader)
print("\n")


print("--- Setting up Trainer ---")
training_args = TrainingArguments(
    output_dir=output_dir,
    optim='adamw_torch',
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3, # Train for only 3 epochs for a quick test
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none",
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader,
    eval_dataset=val_loader
)

print("--- Starting Training ---")
trainer.train()

print("--- Test Complete ---")

"""

--- Initializing Model ---


/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:79: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from '/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/mrna_5utr_model' to '/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/mrna_3utr_model'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:79: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from '/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/mrna_3utr_model' to '/content/drive/MyDrive/Readthrough_project/mRNA-LM-original/data/models/CodonBERT'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


--- Building Dataset ---

--- COLUMNS BEFORE .map() ---
Dataset({
    features: [],
    num_rows: 0
})


--- Mapping Dataset ---

--- COLUMNS AFTER .map() ---
Dataset({
    features: [],
    num_rows: 0
})


--- Setting up Trainer ---


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'